Day 13 — Saturday Build Day. NEEF Statement Analyzer.

In [2]:
import pandas as pd

try:
    data = pd.read_excel("data/neef/fdh_statement.xlsx", index_col=False)
    print(data.isnull().sum())


except ZeroDivisionError as e:
    print(f"something happened {e}")

Booking date        0
Value date          0
Description         0
Payment details     0
Reference           0
Debit              58
Credit              3
Balance             0
dtype: int64


In [3]:
def clean_data():
    data.fillna({'Credit': 0}, inplace=True)
    data.fillna({'Debit': 0}, inplace=True)
    data['Balance'] = data['Balance'].ffill()
    data['Payment details'] = data['Payment details'].str.strip()

clean_data()
data

,Booking date,Value date,Description,Payment details,Reference,Debit,Credit,Balance
0,2026-01-04,2026-01-04,Agent Deposit,AGT18619557<Payer Details- AGENT CASH DEPOSIT<...,FT26091NCVTT\FDH,0.0,120000.0,2114757.25
1,2026-01-04,2026-01-04,Agent Deposit,AGT18614166<Payer Details- AGENT CASH DEPOSIT<...,FT2609178KBY\FDH,0.0,500000.0,2614757.25
2,2026-01-04,2026-01-04,Cash Deposit,YOUNGSON CHILINDA LOAN REPAY ID 204<3871,FT260913LPJ7,0.0,900000.0,3514757.25
3,2026-01-04,2026-01-04,Cash Deposit,MARY KUMWENDA LOAN REPAY ID 0260000<174,FT260913PLFV,0.0,100000.0,3614757.25
4,2026-01-04,2026-01-04,Agent Deposit,AGT18617205<Payer Details- AGENT CASH DEPOSIT<...,FT260913NKMM\FDH,0.0,100000.0,3714757.25
...,...,...,...,...,...,...,...,...
56,2026-04-07,2026-04-07,Cash Deposit,BESTER MVALO,FT26097MSMQ6\MZM,0.0,100000.0,9873557.25
57,2026-04-07,2026-04-07,Online Banking Transfer,FTR TO 8603<FTR TO 8603,FT26097BR0CF\FDH,9000000.0,0.0,873557.25
58,2026-04-07,2026-04-07,Agent Deposit,AGT18610053<Payer Details- AGENT CASH DEPOSIT<...,FT26097F2PSK\FDH,0.0,1220000.0,2093557.25
59,2026-04-07,2026-04-07,Agent Deposit,AGT18619720<Payer Details- AGENT CASH DEPOSIT<...,FT26097PRLYZ\FDH,0.0,50000.0,2143557.25


In [4]:
import csv

def agents():
    with open("data/agents.csv") as row:
        return list(csv.DictReader(row))

In [6]:

ref_list = ["FDH", "MZM", "RPY"]
agent_codes = agents()
description = ["Cash Deposit", "Online Banking Transfer"]

def filter_by_reference(statement, code_list):
    codes = tuple(code for code in code_list)
    pattern = "|".join(codes)

    return statement[statement['Reference'].str.contains(pattern, na=False)]

def filter_by_description(statement, desc):
    des = tuple(d for d in desc)
    des_pattern = "|".join()
    return statement[statement['Description'].str.contains(des_pattern)]

def filter_by_agent(statement, agent_list):
    codes = tuple(agent['agentCode'] for agent in agent_list)
    pattern = "|".join(codes)

    return statement[statement["Payment details"].str.contains(pattern, na=False)]

def filter_combined(statement, desc, ref_code, agent_list):
    ref = tuple(code for code in ref_code)
    a_codes = tuple(agent['agentCode'] for agent in agent_list)
    des = tuple(d for d in desc)


    ref_pattern = "|".join(ref)
    agent_pattern = "|".join(a_codes)
    des_pattern = "|".join(des)

    return statement[
        statement['Description'].str.contains(des_pattern) |
        statement['Reference'].str.contains(ref_pattern) |
        statement['Payment details'].str.contains(agent_pattern)
    ]
# filter_by_reference(data, filter_list)
# filter_by_description(data, "Cash Deposit")
# filter_by_agent(data, agent_codes)
filted_data = filter_combined(data, description, ref_list, agent_codes)

In [7]:
def print_totals(statement):
    total_ts = len(statement)
    total_credit = statement['Credit'].sum()
    total_debit = statement['Debit'].sum()

    print(f"Total transactions: {total_ts:,.2f}")
    print(f"Total credits: {total_credit:,.2f}")
    print(f"Total debits: {total_debit:,.2f}")

    print("All Transactions")
    for _, row in statement.iterrows():
        print(f"{row['Value date']} | {row['Payment details']} | {row['Credit']}")

print_totals(filted_data)

Total transactions: 60.00
Total credits: 23,598,800.00
Total debits: 24,000,000.00
All Transactions
2026-01-04 00:00:00 | AGT18619557<Payer Details- AGENT CASH DEPOSIT<MAJIGA<0997014399 Reference- | 120000.0
2026-01-04 00:00:00 | AGT18614166<Payer Details- AGENT CASH DEPOSIT<WILLIAM PHIRI<0994819314 Reference- | 500000.0
2026-01-04 00:00:00 | YOUNGSON CHILINDA LOAN REPAY ID 204<3871 | 900000.0
2026-01-04 00:00:00 | MARY KUMWENDA LOAN REPAY ID 0260000<174 | 100000.0
2026-01-04 00:00:00 | AGT18617205<Payer Details- AGENT CASH DEPOSIT<MUHUJU CLUB<0996464877 Reference- | 100000.0
2026-01-04 00:00:00 | AGT18610192<Payer Details- AGENT CASH DEPOSIT<TIKOLELANEKO NDABANDABA Reference- | 200000.0
2026-01-04 00:00:00 | AGT18617205<Payer Details- AGENT CASH DEPOSIT<CHARLES KANYINJI<0995910857 Reference- | 500000.0
2026-01-04 00:00:00 | AGT18617598<Payer Details- AGENT CASH DEPOSIT<MANYAMULA COMSIP 1 Reference- | 290000.0
2026-01-04 00:00:00 | AGT18610192<Payer Details- AGENT CASH DEPOSIT<TIGWIRIZ

In [8]:
def export_data(data, name):
    data.to_excel(f"data/clean_data/{name}.xlsx", index=False)
    print("data exported successfully...")

export_data(filted_data, "myclean")

data exported successfully...
